In [24]:
from openai import OpenAI
import re
import os

from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

In [25]:
class Agent:
    """
    A class representing an AI Agent that can engage in conversations using OpenAI's API.
    """

    def __init__(self, system=""):
        """
        Initialize the AI Agent with an optional system message.

        Args:
            system (str): The system message to initialize the agent with.
        """
        self.system = system
        self.messages = []
        if self.system:
            # If a system message is provided, add it to the messages history
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        """
        Allow the agent to be called with a message.

        Args:
        message (str): The user's input message.

        Returns:
        str: Agents response to the user's message.
        """
        # Add the user message's to the conversation history
        self.messages.append({"role": "user", "content": message})
        # Execute the conversation and get the results
        result = self.execute()
        # Add the assistant's response to the conversation history
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        """
        Execute the conversation using entire conversation history to the OpenAI API
        Returns:
        str : The content of the model's response.
        """
        # Send the entire conversation history to the OpenAI API
        completion = client.chat.completions.create(
            model="gpt-4.1-mini",
            temperature=0.7,
            messages=self.messages
        )
        return completion.choices[0].message.content

In [26]:
# Price lookup for fruits
fruit_price = {
    "apple": 1.5,
    "banana": 1.2,
    "orange": 1.3,
    "grapes": 2.0
    }

# Function to calculate the specific fruit price
def get_fruit_price(fruit):
    if fruit in fruit_price:
        return f" The price of a {fruit} is ${fruit_price[fruit]}"
    else:
        return f"Sorry, we don't have the price for {fruit}."

def calculate_total_price(fruits):
    total = 0
    fruit_list = fruits.split(", ")
    for item in fruit_list:
        fruit, quantity = item.split(":")
        quantity = int(quantity)
        if fruit in fruit_price:
            total += fruit_price[fruit] * quantity
        else:
            return f"Sorry, we don't have the price for {fruit}."
    return f"The total price for the selected fruits is ${total}"

# mapping actions to functions
action_mapping = {
    "get_fruit_price": get_fruit_price,
    "calculate_total_price": calculate_total_price
}
#print(action_mapping["get_fruit_price"]("apple"))
#print(action_mapping["calculate_total_price"]("apple:2"))

In [27]:
# Run the query
import re

action_re = re.compile(r'^Action: (\w+): (.*)$')


def query(question):
    system = (
        "You are a fruit store assistant. For a single fruit price, respond "
        "with exactly: Action: get_fruit_price: <fruit>. For a total price, "
        "respond with exactly: Action: calculate_total_price: "
        "<fruit>:<quantity>, <fruit>:<quantity>. Use lowercase fruit names "
        "and do not include any extra text."
    )
    bot = Agent(system)
    result = bot(question)
    print(result)
    actions = [
        action_re.match(line)
        for line in result.splitlines()
        if action_re.match(line)
    ]
    if actions:
        action, action_input = actions[0].groups()
        if action not in action_mapping:
            raise ValueError(f"Unknown action: {action}: {action_input}")

        print(f" --running {action} {action_input}")
        observation = action_mapping[action](action_input)
        print(f" --observation: {observation}")
    else:
        return
    

In [38]:
# Run this after adding OpenAI API credits:
# query("What is the price of an orange?")

# Local lookup works without an API call.
print(action_mapping["get_fruit_price"]("orange"))
print(action_mapping["calculate_total_price"]("apple:2, orange:3"))
print(action_mapping["calculate_total_price"]("grapes:7, banana:5"))

 The price of a orange is $1.3
The total price for the selected fruits is $6.9
The total price for the selected fruits is $20.0


In [7]:
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph
builder = StateGraph(dict)
def my_node(state: dict, config: RunnableConfig):
    return {"results": f"My name is {state['input']}!"}
# The Second argument is optional
def my_other_node(state: dict):
    return state
builder.add_node("my_node", my_node)
builder.add_node("my_other_node", my_other_node)
builder.add_edge("my_node", "my_other_node")
builder.set_entry_point("my_node")
builder.set_finish_point("my_other_node")

graph = builder.compile()
#graph.invoke({"input": "Mallika"})

Structured Outputs

In [1]:
{
    "entities": [
        {
            "name": "John Deo", "type": "person"
        },
        {
            "name": "OpenAI", "type": "organization"
        },
        {
            "name": "San Francisco", "type": "location"
        },
    ],
    "Summary": "John Deo from OpenAI visited San Francisco to discuss Advancements in AI."
}

{'entities': [{'name': 'John Deo', 'type': 'person'},
  {'name': 'OpenAI', 'type': 'organization'},
  {'name': 'San Francisco', 'type': 'location'}],
 'Summary': 'John Deo from OpenAI visited San Francisco to discuss Advancements in AI.'}

String Prompt Template

In [2]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a joke about {topic}")
prompt_template.invoke({"topic": "cats"})

StringPromptValue(text='Tell me a joke about cats')

Chat Prompt Template

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Tell me a joke about {topic}")
])
prompt_template.invoke({"topic": "cats"})

llm = ChatOpenAI(
    model="llama3.2:3b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    temperature=0.7,
)
parser = StrOutputParser()
chain = prompt_template | llm | parser
chain.invoke({"topic": "cats"})

'Why did the cat join a band?\n\nBecause it wanted to be the purr-cussionist.'

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
_ = load_dotenv()
model = ChatOpenAI(
    model="llama3.2:3b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)
system_template = """
You are a trip planner expert. Help me plan a trip to {destination}.
Consider my preferences and provide a detailed itinerary.
"""
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", "Plan a trip to {destination} with preferences {preferences}")
])
parser = StrOutputParser()
trip_planner_chain = prompt_template | model | parser

def plan_trip(destination, preferences):
    input_data = {"destination": destination, "preferences": preferences}
    return trip_planner_chain.invoke(input_data)

In [16]:
# helper function to render Markdown
from IPython.display import Markdown, display

def render_markdown(md_string):
    display(Markdown(md_string))

In [19]:
result = plan_trip("Paris", {"preferences": "sightseeing, food"})
render_markdown(result)

Paris, the City of Light! I'd be delighted to plan a trip to this beautiful city for you. Based on your preferences for sightseeing and food, I've created a 7-day itinerary that will take you to the most iconic landmarks, delicious culinary experiences, and hidden gems.

**Day 1: Arrival and Montmartre Exploration**

* Arrive at Charles de Gaulle Airport and take a taxi or train to your accommodation
* Check-in to your hotel and freshen up
* Head to the historic neighborhood of Montmartre, known for its bohemian vibe, street artists, and stunning views of the city
* Visit the Basilique du Sacré-Cœur, a beautiful white church perched on a hill
* Wander through the charming streets and alleys of Montmartre, exploring the Place du Tertre, and stopping for a coffee or snack at a quaint café
* Enjoy dinner at a traditional French bistro, such as Le Comptoir du Relais or Chez L'Ami Jean

**Day 2: Iconic Landmarks**

* Start the day with a visit to the iconic Eiffel Tower (buy tickets in advance to avoid long queues)
* Take the stairs or elevator to the top for breathtaking views of the city
* Next, head to the nearby Champ de Mars park for a picnic lunch or snack
* After lunch, visit the world-famous Louvre Museum (buy tickets in advance to skip the lines)
* Spend the late afternoon exploring the trendy Marais neighborhood, known for its fashionable boutiques, art galleries, and trendy restaurants
* Enjoy dinner at a Michelin-starred restaurant, such as Le Grand Vefour or L'Atelier de Joël Robuchon

**Day 3: Foodie Heaven**

* Spend the day exploring the culinary delights of Paris
* Start with a visit to the historic market, Marché d'Aligre, where you can sample artisanal cheeses, charcuterie, and fresh produce
* Head to the charming neighborhood of Le Marais for lunch at a trendy bistro, such as Le Loir dans la Theière or Le Bistrot Paul Bert
* Next, visit the iconic Ladurée patisserie for a taste of French macarons and other sweet treats
* Enjoy dinner at a classic Parisian restaurant, such as Le Petit Châtelet or Le Petit Nice

**Day 4: Palace of Versailles**

* Take a day trip to the Palace of Versailles, a former royal residence with opulent gardens and fountain shows
* Explore the palace's opulent interiors, including the Hall of Mirrors and the Royal Chapel
* Stroll through the beautiful gardens, including the Orangery and the Jardin d'Hiver
* Return to Paris in the evening and enjoy dinner at a local restaurant, such as Le Comptoir du Relais or Le Pied de Cochon

**Day 5: Art and Culture**

* Spend the day exploring Paris's vibrant art scene
* Visit the Musée d'Orsay, home to an impressive collection of Impressionist and Post-Impressionist art
* Next, head to the iconic Pompidou Center, a modern art museum with a colorful exterior and innovative exhibitions
* Enjoy lunch at a trendy café, such as Le Grenier de Notre-Dame or Le Café de la Paix
* Spend the evening exploring the lively neighborhood of Belleville, known for its street art, independent shops, and multicultural restaurants

**Day 6: Seine River Cruise and Montparnasse**

* Take a scenic Seine River cruise to admire the city's landmarks from a different perspective
* Visit the iconic Notre-Dame Cathedral and the Sainte-Chapelle, a stunning Gothic chapel with intricate stained glass
* Explore the historic neighborhood of Montparnasse, known for its bohemian vibe, charming cafes, and stunning views of the city
* Enjoy dinner at a classic Parisian restaurant, such as Le Grand Vefour or Le Petit Châtelet

**Day 7: Departure**

* Spend the morning shopping for souvenirs or exploring a local market
* Depart for the airport and head home, filled with memories of your Parisian adventure!

**Accommodation**

I recommend staying in the heart of the city, in a charming neighborhood such as Le Marais, Montmartre, or the Latin Quarter. Some top-rated hotels include:

* Hotel Le Walt (Le Marais)
* Hotel des Grands Boulevards (Les Halles)
* Hotel Le Pigalle (Montmartre)

**Budget Breakdown**

Here's a rough estimate of the costs for the 7-day trip:

* Accommodation: €2,000 - €3,000 (approximately $2,200 - $3,300 USD)
* Transportation: €200 - €300 (approximately $220 - $330 USD)
* Food and drink: €800 - €1,200 (approximately $880 - $1,320 USD)
* Attractions and activities: €500 - €800 (approximately $550 - $880 USD)
* Total: €3,300 - €5,300 (approximately $3,600 - $5,800 USD)

**Tips and Recommendations**

* Buy tickets in advance for popular attractions to avoid long queues
* Try to avoid visiting during peak tourist season (June-August) for smaller crowds
* Don't be afraid to try new foods and drinks, such as escargots, croissants, or French champagne
* Explore the city's charming neighborhoods and discover hidden gems
* Take a scenic Seine River cruise to admire the city's landmarks from a different perspective

Bon voyage et bon appétit!